# Cross-Dataset Evaluation / 跨数据集评估

`04_leakage_audit` 的结论是：ISOT 里的真新闻全部来自路透社，模型靠"路透社文风"就能拿到高分，清洗也无法消除这一点。本 notebook 回答下一个问题：**在 ISOT 上训练的模型，换一批来源后还剩多少能力？**

- **Part A（本部分，步骤 1a）**：选择外部数据集，下载、校验，并按和 ISOT 完全相同的方式处理
- **Part B（步骤 1c，之后补充）**：跨数据集实验

`04_leakage_audit` showed that ISOT's real news is all Reuters, so a model can score highly on "Reuters style" — something cleaning cannot remove. This notebook asks: **how much ability does an ISOT-trained model keep on data from different sources?** Part A (this part) selects, downloads, verifies and processes the external data exactly like ISOT; Part B (later) runs the experiments.

## 1. Setup / 环境

In [1]:
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path.cwd().parent))

from src.utils import set_seed
from src.data import load_dataset, to_model_text
from src.external import (load_welfake_raw, load_welfake_external, load_welfake_report,
                          load_liar, verify_welfake_label_direction, _fingerprints, _key_set,
                          _english_share, MIN_ENGLISH_SHARE, WELFAKE_MD5, LIAR_MD5, EXTERNAL_DIR, _md5)
from src.data import load_raw
from src import probes

set_seed(42)
pd.set_option('display.max_colwidth', 100)

/Users/sl1425/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


## 2. Choosing the external dataset / 选择外部数据集

理想的外部数据集应满足三个条件：**全文新闻**（和 ISOT 格式一致）、**真新闻不只来自路透社**、**可以合法使用**。检查过的三个候选：

| 数据集 | 内容 | 许可 | 结论 |
|---|---|---|---|
| FakeNewsNet（Hugging Face 版） | **只有标题和链接**，没有正文；大部分是 GossipCop 娱乐新闻 | CC | ❌ 不适用：没有正文，完整版需要逐条抓取，很多链接已失效 |
| LIAR（Wang, 2017） | 12,836 条 PolitiFact 逐条核查的短陈述（中位数 17 个词） | 仅限研究用途 | ⚠️ 次要：标签直接反映真假，但只有一句话，和长文章格式差异太大 |
| **WELFake**（Verma et al., 2021） | 72,134 篇全文新闻，合并了 Kaggle、McIntire、Reuters、BuzzFeed 四个数据集 | **CC-BY-4.0** | ✅ 主要：**去掉其中的 ISOT 部分后**，真新闻来自《纽约时报》、Breitbart 等，不是路透社 |

An external set should be **full-text**, have **real news beyond Reuters**, and be **legally usable**. FakeNewsNet (HF) has titles only; LIAR is one-sentence statements (secondary); **WELFake minus its ISOT portion** is the primary choice (CC-BY-4.0).

> **重要说明 / Caveat**：WELFake 的标签同样是 **按来源网站** 打的（例如 Breitbart 整体被标为"真"），不是逐篇核查。所以它检验的是"路透社文风学到的东西能否迁移到其他来源"，而不是"模型能否判断真假"。后者需要 LIAR 这类逐条核查的数据，或者步骤 5 的证据核查方法。
> WELFake labels are also **per source site** (Breitbart is "real" wholesale), so it tests whether what was learned from Reuters style transfers to other sources — not truthfulness itself.

数据来源 / Sources:
- WELFake: Verma, Agrawal & Prodan (2021), *IEEE Transactions on Computational Social Systems*. Zenodo, [doi:10.5281/zenodo.4561253](https://doi.org/10.5281/zenodo.4561253), CC-BY-4.0
- LIAR: Wang (2017), ["Liar, Liar Pants on Fire"](https://aclanthology.org/P17-2067/), ACL 2017. Research use only.

## 3. Download & checksum / 下载与校验

文件下载到 `data/external/`（不提交到 git）。WELFake 的 MD5 来自 Zenodo 官方 API；LIAR 官方没有公布校验值，记录的是第一次下载时的 MD5，保证以后拿到的是同一份文件。

Files go to `data/external/` (not in git). The WELFake MD5 is Zenodo's official value; LIAR has none, so the MD5 from the first download is pinned.

In [2]:
for name, expected in [('WELFake_Dataset.csv', WELFAKE_MD5), ('liar_dataset.zip', LIAR_MD5)]:
    path = EXTERNAL_DIR / name
    actual = _md5(path)
    print(f'{name:22s} {path.stat().st_size / 1e6:7.1f} MB  md5 {actual}  '
          f'{"OK" if actual == expected else "MISMATCH"}')

WELFake_Dataset.csv      245.1 MB  md5 73c9675a4b3d09f86a6933d0b8d7d908  OK
liar_dataset.zip           1.0 MB  md5 437541ca9957e993634f9c1e71a26748  OK


## 4. Label direction check / 标签方向检查

Zenodo 页面写的是 "0 = fake and 1 = real"。WELFake 里包含 ISOT 的原文，可以用它们来验证：**ISOT 真新闻在 WELFake 里被标成了什么？**

`load_welfake_raw()` 已经按下面的验证结果把标签转换成本项目的约定（1 = 真，0 = 假）。转换正确时，下表应该几乎只有对角线上有数字。

The Zenodo page says "0 = fake and 1 = real". Since WELFake contains ISOT articles, we can check what ISOT's real news is labelled as. `load_welfake_raw()` converts to this project's convention (1 = real); after a correct conversion the table below is diagonal.

In [3]:
welfake_raw = load_welfake_raw()
isot_raw = load_raw()
verify_welfake_label_direction(welfake_raw, isot_raw)

WELFake label (converted),0,1
ISOT label,,
0,22641,1
1,0,21416


转换后两边几乎完全一致（44,058 篇里只有 1 篇不同）。这说明 **原始 CSV 实际上是 1 = 假、0 = 真**，正好和 Zenodo 页面的说明相反；论文中"37,106 篇假新闻"的数量也和原始 `label == 1` 的数量一致。如果照页面说明直接使用，所有跨数据集结果都会完全反过来，而且很难察觉。`verify_welfake_label_direction()` 每次构建时都会重新检查，一致率低于 99.9% 就会报错。

After conversion the labels agree on all but 1 of 44,058 articles: the **raw CSV is actually 1 = fake, 0 = real**, the opposite of the Zenodo page (the paper's "37,106 fake" equals the raw count of `label == 1`). Used as documented, every cross-dataset result would be silently inverted. The check re-runs on every build and fails below 99.9% agreement.

## 5. Building "WELFake minus ISOT" / 构建外部测试集

处理方式和 ISOT 完全相同（同一套 `clean_title` / `clean_text` / `to_model_text`），另外有三项外部数据专用的步骤：
1. 删除标题末尾的出版方后缀，例如 " - The New York Times"（真新闻 63%、假新闻 2.6%，和 ISOT 的 "[VIDEO]" 一样是来源标签）
2. 删除非英文文章
3. 删除和 ISOT **任何部分**（train + val + test）有关联的文章：先用原文比对，再比较五种指纹

Processing is identical to ISOT, plus three external-only steps: strip publisher suffixes, drop non-English articles, and drop anything linked to ANY part of ISOT (raw-text match, then five fingerprints).

In [4]:
welfake = load_welfake_external()
report = load_welfake_report()

steps = pd.DataFrame({k: report[k] for k in [
    'removed_isot_copies_raw_match', 'removed_short_body', 'removed_non_english',
    'removed_exact_duplicates', 'removed_isot_overlap_fingerprint', 'final_by_label']}).T
steps.index = ['ISOT 副本（原文比对）/ ISOT copies (raw match)',
               '正文 < 20 词 / body < 20 words',
               '非英文 / non-English',
               '完全重复 / exact duplicates',
               'ISOT 近似副本（指纹）/ ISOT near-copies (fingerprints)',
               '最终 / final']
print(f"Raw WELFake rows: {report['raw_total']:,}   conflicting-label rows removed: "
      f"{report['removed_conflicting_label_rows']}")
steps

Raw WELFake rows: 72,134   conflicting-label rows removed: 0


,fake,real
ISOT 副本（原文比对）/ ISOT copies (raw match),22641,21417
正文 < 20 词 / body < 20 words,1429,159
非英文 / non-English,447,2
完全重复 / exact duplicates,2576,16
ISOT 近似副本（指纹）/ ISOT near-copies (fingerprints),69,45
最终 / final,9944,13389


几点说明 / Notes:
- **ISOT 副本占了 WELFake 的六成以上**（44,058 / 72,134）。如果不去掉，跨数据集测试其实就是在 ISOT 上测试。
- **原文比对之后，指纹又找出了 114 篇** 和 ISOT 相关的近似副本（原文有细微差异、或者换了标题 / 开头）。只做原文比对是不够的。
- **非英文文章几乎全是假新闻**（447 篇假、2 篇真），保留的话"不是英文"本身就会成为一个假新闻信号。
- 最终 **23,333 篇**，真新闻占 57%。

- **ISOT copies make up over 60% of WELFake**; without removing them, "cross-dataset" would really be testing on ISOT.
- **Fingerprints catch 114 more** near-copies after the raw match — a raw match alone is not enough.
- **Non-English articles are almost all fake**; kept, "not English" would itself signal fake.
- Final: **23,333 articles**, 57% real.

## 6. Verification / 验证

三项独立检查 / Three independent checks:
1. 和清洗后的 ISOT（全部 37,982 篇）比较五种指纹，**必须全部为 0**
2. 不应残留非英文文章
3. 残留的来源格式痕迹，和 ISOT 清洗后的情况并排比较

In [5]:
isot = load_dataset()
fp_isot = _fingerprints(isot['title'], isot['text'], isot['model_text'])
fp_ext = _fingerprints(welfake['title'], welfake['text'], welfake['model_text'])
overlap = {name: int(fp_ext[name].isin(_key_set(fp_isot[name])).sum()) for name in fp_ext}
print('1) external articles sharing a fingerprint with ISOT:', overlap)
print('2) non-English articles left:',
      int((_english_share(welfake['model_text']) < MIN_ENGLISH_SHARE).sum()))

Loaded cached dataset: 37,982 rows (clean_dataset.csv)


1) external articles sharing a fingerprint with ISOT: {'exact': 0, 'title': 0, 'lead': 0, 'tail': 0, 'middle': 0}


2) non-English articles left: 0


In [6]:
pd.concat({'ISOT (cleaned)': probes.artefact_rates(isot),
           'WELFake-external': probes.artefact_rates(welfake)}, axis=1)

ISOT (cleaned)        WELFake-external       
                                      fake % real %           fake % real %
Reuters 字样 / "Reuters"                   0.0    0.1              0.4    0.1
开头署名 / dateline "(Reuters) -"            0.0    0.0              0.0    0.0
空括号 / empty "()"                         0.0    0.0              0.0    0.0
记者署名 / "(Reporting by"                   0.0    0.0              0.0    0.0
图片来源 / "Featured image"                  0.0    0.0              0.0    0.0
Getty Images                             0.1    0.0              0.4    0.2
转载来源 / "Via:" / "Read more"              0.0    0.0              0.0    0.0
推文链接 / pic.twitter.com                   0.0    0.0              0.0    0.0
URL                                      0.0    0.0              0.0    0.1
@用户名 / @handle                           0.0    0.0              0.0    0.0
弯引号 / curly quotes                       0.0    0.0              0.0    0.0
不换行空格 / \xa0                             0.0    0.0              0.0    0.0
句号后缺空格 / "word.Word"                     0.0    0.0              0.0    0.0
标题方括号 / title "[...]"                    0.0    0.0              0.0    0.0
标题 VIDEO / WATCH                         3.0    0.3              1.2    0.7
标题感叹号 / title "!"                        8.4    0.1              4.8    0.4
文风 / "said"                             57.7   93.7             44.7   82.4
文风 / "U.S."                             16.6   60.2             19.4    7.8

格式类痕迹在两个数据集中都接近 0，所以跨数据集测试不会被这些痕迹干扰。

值得注意的是最后两行的 **文风** 特征：
- **"said"** 在两个数据集中都是真新闻多得多（ISOT 94% vs 58%；WELFake 82% vs 45%），因为《纽约时报》和路透社一样，都是大量引用消息来源的新闻写法。所以 ISOT 学到的 "said" 线索，在 WELFake 上 *仍然有用*。
- **"U.S."** 则正好 **反过来**：在 ISOT 里是真新闻多（60% vs 17%），在 WELFake 里是假新闻多（19% vs 8%）。不过 `model_text` 会把 "U.S." 和 "US" 合并，所以这个具体差异不会直接进入模型。

这说明文风线索换了数据集后，**有的能迁移，有的会失效甚至反向**。所以跨数据集的结果不会是简单的"全对"或"全错"，Part B 需要结合特征权重一起解读。

Formatting artefacts are ~0 in both sets. Style is mixed: **"said"** favours real news in both datasets (NYT, like Reuters, is attribution-heavy), so that cue *still helps* on WELFake; **"U.S."** flips direction between the two. Some style cues transfer and some reverse, so Part B results must be read together with feature weights.

## 7. Where does the external real news come from? / 外部真新闻的来源

清洗时出版方后缀已经被删掉，这里回到原始标题，统计最终保留的真新闻来自哪些出版方。

Publisher suffixes were stripped during cleaning, so we count them on the raw titles of the articles that were kept.

In [7]:
raw_titles = welfake_raw.set_index('doc_id').loc[welfake['doc_id'], 'title']
publisher = raw_titles.str.extract(r' - (The New York Times|Breitbart)\s*$')[0].fillna('no suffix (other / unknown)')
pd.crosstab(publisher.values, welfake['label'].map({0: 'fake', 1: 'real'}).values,
            margins=True, margins_name='total')

col_0,fake,real,total
row_0,,,
Breitbart,1,2334,2335
The New York Times,0,6219,6219
no suffix (other / unknown),9943,4836,14779
total,9944,13389,23333


外部真新闻里，46%（6,219 篇）来自《纽约时报》，17%（2,334 篇）来自 Breitbart，其余 36% 没有出版方后缀（来自 McIntire 和 BuzzFeed 数据集中的多家媒体）。**没有一篇来自路透社**，这正是我们需要的：ISOT 模型如果只学会了"路透社文风"，在这里就会失效。

46% of the external real news is NYT (6,219) and 17% Breitbart (2,334); the remaining 36% carries no suffix (multiple outlets from the McIntire and BuzzFeed subsets). **None is Reuters** — exactly what is needed to test whether an ISOT model learned more than Reuters style.

## 8. LIAR overview / LIAR 概况

LIAR 作为 **次要** 外部测试集：每条都是 PolitiFact 编辑逐条核查过的陈述，标签直接反映真假，但只有一句话。6 级标签按文献中最常见的方式转为二分类：true / mostly-true / half-true → 真，barely-true / false / pants-fire → 假。`label6` 保留了原始 6 级标签，以后可以尝试其他划分方式（例如把 half-true 去掉）。

LIAR is the **secondary** external set: individually fact-checked, but one sentence each. The 6 labels are binarised the most common way (true / mostly-true / half-true → real); `label6` keeps the original labels for other mappings later.

In [8]:
liar = load_liar()
print(liar.groupby('split')['label'].agg(n='size', real_share='mean').round(3))
print()
print('Original 6-way labels:', liar['label6'].value_counts().to_dict())
print('Median length (words):  LIAR', int(liar['text'].str.split().str.len().median()),
      '| WELFake-external', int(welfake['text'].str.split().str.len().median()),
      '| ISOT', int(isot['text'].str.split().str.len().median()))

           n  real_share
split                   
test    1283       0.567
train  10269       0.562
val     1284       0.520

Original 6-way labels: {'half-true': 2638, 'false': 2511, 'mostly-true': 2466, 'barely-true': 2108, 'true': 2063, 'pants-fire': 1050}


Median length (words):  LIAR 17 | WELFake-external 614 | ISOT 368


## 9. Part A summary / A 部分小结

| | WELFake-external（主要） | LIAR（次要） |
|---|---|---|
| 规模 | 23,333 篇（真 57%） | 12,836 条（官方划分 10,269 / 1,284 / 1,283） |
| 格式 | 全文新闻，和 ISOT 一致 | 一句话陈述（中位数 17 个词） |
| 真新闻来源 | 《纽约时报》、Breitbart 等，**没有路透社** | PolitiFact 核查的各类陈述 |
| 标签依据 | 按来源网站 | **逐条核查** |
| 和 ISOT 的重叠 | 0（五种指纹） | 不同类型的数据，不存在全文重叠 |
| 许可 | CC-BY-4.0 | 仅限研究用途 |

**处理过程中发现并解决的问题 / Issues found and fixed:**
1. WELFake 的标签方向和官方说明 **相反**，已转换，并在每次构建时自动验证
2. WELFake 六成以上是 ISOT 原文，已全部删除，包括 114 篇近似副本
3. 外部真新闻标题带出版方后缀（63%），属于来源泄漏，已删除
4. 非英文文章几乎全是假新闻（447 / 449），已删除
5. pandas `isin()` 会把 NaN 和 NaN 视为相等，可能把大量文章误判为重叠，已修复并加了单元测试

**下一步（Part B，步骤 1c）**：用 TF-IDF + 逻辑回归做两个实验：
- **ISOT → WELFake-external**：在 ISOT 上训练，在外部数据上测试
- **WELFake-external → WELFake-external**：在外部数据内部训练和测试，作为这个数据集本身的难度参照

如果第一个分数远低于第二个，就说明 ISOT 模型学到的主要是路透社文风，无法迁移到其他来源。